# Multi-Block PLS

MB-PLS handles X data from multiple sources (blocks) simultaneously, assigning a weight to each block. Useful when variables come from fundamentally different measurement sources (e.g., NIR + process + lab).

In [1]:
import pandas as pd
import numpy as np
import pyphi.calc as phi
import pyphi.plots as pp
from bokeh.io import output_notebook
output_notebook(hide_banner=True)
import pyphi.plots as _ppmod
_ppmod.output_file = lambda *a, **kw: None


Will be using the NEOS server in the absence of IPOPT and GAMS


## Load Data

`XMB` must be a **dict** mapping block names to DataFrames (each with obs-ID first column).

In [2]:
xb = {
    f'X{i}': pd.read_excel('../data/MBDataset.xlsx', sheet_name=f'X{i}')
    for i in range(1, 7)
}
y_data = pd.read_excel('../data/MBDataset.xlsx', sheet_name='Y')
for k, v in xb.items():
    print(k, v.shape)
print('Y:', y_data.shape)


X1 (50, 6)
X2 (50, 5)
X3 (50, 4)
X4 (50, 4)
X5 (50, 5)
X6 (50, 5)
Y: (50, 5)


## Build MB-PLS Model

In [3]:
mbpls_obj = phi.mbpls(xb, y_data, 2)
print('Keys:', list(mbpls_obj.keys()))


phi.pls using NIPALS executed on: 2026-03-27 00:03:08.952397
# Iterations for LV #1:  36
# Iterations for LV #2:  22
--------------------------------------------------------------
LV #     Eig       R2X       sum(R2X)   R2Y       sum(R2Y)
LV #1:    0.739    0.143     0.143      0.090     0.090
LV #2:    0.453    0.090     0.233      0.088     0.178
--------------------------------------------------------------
Keys: ['T', 'P', 'Q', 'W', 'Ws', 'U', 'r2x', 'r2xpv', 'mx', 'sx', 'r2y', 'r2ypv', 'my', 'sy', 'var_t', 'obsidX', 'varidX', 'obsidY', 'varidY', 'T2', 'T2_lim99', 'T2_lim95', 'speX', 'speX_lim99', 'speX_lim95', 'speY', 'speY_lim99', 'speY_lim95', 'type', 'x_means', 'x_stds', 'y_means', 'y_stds', 'Xblk_scales', 'Yblk_scales', 'Wsb', 'Wt', 'r2pbX', 'r2pbXc', 'Xblocknames']


## Score and Loadings Plots

In [4]:
pp.score_scatter(mbpls_obj, [1, 2])
pp.loadings(mbpls_obj)
pp.weighted_loadings(mbpls_obj)


## Block-Level Diagnostics

In [5]:
pp.r2pv(mbpls_obj)
pp.mb_r2pb(mbpls_obj)
pp.mb_weights(mbpls_obj)
pp.mb_vip(mbpls_obj)
pp.vip(mbpls_obj)


## Prediction on New Data

Pass the same dict structure to `pls_pred`.

In [6]:
preds = phi.pls_pred(xb, mbpls_obj)
print('Yhat shape:', preds['Yhat'].shape)


Yhat shape: (50, 4)
